## Imports

In [1]:
import gc
import re
from pathlib import Path

import numpy as np
import pandas as pd
import tifffile
from careamics import CAREamist
from IPython.display import display
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

/opt/.pixi/envs/napari-careamics-default/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Helper functions

In [2]:
def image_metrics(target, image):
    """Measure SSIM, NRMSE, and PSNR after matching image intensity to its target."""
    target = np.asarray(target, dtype=np.float32).squeeze()
    image = np.asarray(image, dtype=np.float32).squeeze()

    if target.ndim != 2 or image.ndim != 2:
        raise ValueError(f"Expected 2D images; got target {target.shape}, image {image.shape}")
    if target.shape != image.shape:
        raise ValueError(f"Image shape mismatch: target {target.shape}, image {image.shape}")
    if not np.isfinite(target).all() or not np.isfinite(image).all():
        raise ValueError("Images contain NaN or infinite pixel values")

    low, high = np.percentile(target, (0.1, 99.9))
    if high <= low:
        raise ValueError("Target image has no intensity range")

    target_norm = (target - low) / (high - low)
    image_centered = image - image.mean()
    target_centered = target_norm - target_norm.mean()

    image_variance = np.var(image_centered)
    if image_variance == 0:
        raise ValueError("Cannot intensity-match a constant image")

    # Affine intensity matching, following the ZeroCostDL4Mic CARE quality-control approach.
    scale = np.cov(image_centered.ravel(), target_centered.ravel())[0, 1] / image_variance
    image_matched = scale * image_centered

    return {
        "SSIM": structural_similarity(
            target_norm,
            image_matched,
            data_range=1.0,
            gaussian_weights=True,
            use_sample_covariance=False,
            sigma=1.5,
        ),
        "NRMSE": np.sqrt(np.mean((target_norm - image_matched) ** 2)),
        "PSNR (dB)": peak_signal_noise_ratio(
            target_norm,
            image_matched,
            data_range=1.0,
        ),
    }


def summarize_metrics(images, targets):
    """Return image count and mean +/- standard deviation for each metric."""
    if len(images) != len(targets):
        raise ValueError(f"Got {len(images)} images for {len(targets)} targets")
    if not images:
        raise ValueError("No images were provided for metric calculation")

    per_image = [image_metrics(target, image) for target, image in zip(targets, images)]
    summary = {"Images": len(per_image)}

    for metric in ("SSIM", "NRMSE", "PSNR (dB)"):
        values = [result[metric] for result in per_image]
        summary[metric] = f"{np.mean(values):.3f} +/- {np.std(values):.3f}"

    return summary

## Load data: set the prepared split to assess

In [3]:
# These paths are relative to the CAREamics/scripts folder.
data_split = "U2OS_splits_n100_seed42"
prepared_dir = Path("../data_careamics") / data_split

test_low_dir = prepared_dir / "test" / "low"
test_high_dir = prepared_dir / "test" / "high"

In [4]:
if not test_low_dir.is_dir() or not test_high_dir.is_dir():
    raise FileNotFoundError(
        f"Expected test low/high folders under {prepared_dir.resolve()}"
    )

test_low_paths = sorted(test_low_dir.glob("*.tif"))
test_high_paths = sorted(test_high_dir.glob("*.tif"))

if not test_low_paths:
    raise FileNotFoundError(f"No test TIFFs found in {test_low_dir.resolve()}")

if [path.name for path in test_low_paths] != [path.name for path in test_high_paths]:
    raise ValueError("Test low/high TIFF filenames do not match")

test_inputs = [tifffile.imread(path) for path in test_low_paths]
test_targets = [tifffile.imread(path) for path in test_high_paths]

print(f"Loaded {len(test_inputs)} test pairs from {prepared_dir.resolve()}")

Loaded 10 test pairs from /storage/research/dsl_vibe_rs/trainings/2026_deep_learning_with_VIBE/CAREamics/data_careamics/U2OS_splits_n100_seed42


In [ ]:
## Find checkpoints, select models, and parse their hyperparameters
models_dir = Path("../models")
available_checkpoints = sorted(models_dir.rglob("*.ckpt"))

if not available_checkpoints:
    raise FileNotFoundError(f"No CAREamics checkpoints found under {models_dir.resolve()}")

# Default: assess the final checkpoint for each training run.
available_last_checkpoints = [
    path for path in available_checkpoints if path.name == "last.ckpt"
]

print("Available checkpoints:")
for checkpoint in available_checkpoints:
    print(f"  {checkpoint.relative_to(models_dir)}")

if not available_last_checkpoints:
    raise FileNotFoundError(f"No last.ckpt files found under {models_dir.resolve()}")

# Change this selection to assess different checkpoints, e.g.
# selected_checkpoints = [available_checkpoints[0], available_checkpoints[3]]
selected_checkpoints = available_last_checkpoints

if not selected_checkpoints:
    raise ValueError("Select at least one checkpoint to assess")
if len(set(selected_checkpoints)) != len(selected_checkpoints):
    raise ValueError("A checkpoint was selected more than once")

missing_checkpoints = [path for path in selected_checkpoints if not path.is_file()]
if missing_checkpoints:
    raise FileNotFoundError(
        f"Selected checkpoint not found: {missing_checkpoints[0].resolve()}"
    )

RUN_NAME_PATTERN = re.compile(
    r"^(?P<training_set>.+)_(?P<backbone>[A-Za-z0-9]+)-finetuned"
    r"_n_epochs(?P<epochs>\d+)_b_size(?P<batch_size>\d+)"
    r"_w_decay(?P<weight_decay>[^_]+)_l_rate(?P<learning_rate>.+)$"
)


def parse_run_name(run_name):
    match = RUN_NAME_PATTERN.fullmatch(run_name)
    if not match:
        raise ValueError(f"Could not parse hyperparameters from model folder: {run_name}")

    params = match.groupdict()
    encoded_decay = params["weight_decay"]

    # The training notebook removes the decimal point: for example, 0.1 becomes "01".
    params["weight_decay"] = (
        float(f"{encoded_decay[0]}.{encoded_decay[1:]}")
        if encoded_decay.startswith("0") and len(encoded_decay) > 1
        else float(encoded_decay)
    )
    params["epochs"] = int(params["epochs"])
    params["batch_size"] = int(params["batch_size"])
    params["learning_rate"] = float(params["learning_rate"])
    return params


model_info = {}
for checkpoint in selected_checkpoints:
    run_name = checkpoint.parent.parent.name
    label = f"{run_name} / {checkpoint.stem}"
    params = parse_run_name(run_name)

    model_info[label] = {
        "Training set": params["training_set"],
        "Backbone": params["backbone"],
        "Epochs": params["epochs"],
        "Batch size": params["batch_size"],
        "Weight decay": params["weight_decay"],
        "Learning rate": params["learning_rate"],
        "Checkpoint": checkpoint.stem,
    }

## Create predictions

In [6]:
# Models are loaded one at a time to avoid keeping all of them in memory.
model_predictions = {}

for checkpoint in selected_checkpoints:
    run_name = checkpoint.parent.parent.name
    label = f"{run_name} / {checkpoint.stem}"
    print(f"Predicting with {label}")

    careamist = CAREamist(source=str(checkpoint))
    predictions = careamist.predict(source=str(test_low_dir))

    if len(predictions) != len(test_targets):
        raise ValueError(
            f"{checkpoint.name} returned {len(predictions)} predictions for "
            f"{len(test_targets)} test targets"
        )

    model_predictions[label] = predictions
    del careamist
    gc.collect()

rows = [
    {
        "Model / input": "Noisy input",
        "Training set": None,
        "Backbone": None,
        "Epochs": None,
        "Batch size": None,
        "Weight decay": None,
        "Learning rate": None,
        "Checkpoint": None,
        **summarize_metrics(test_inputs, test_targets),
    }
]

for label, predictions in model_predictions.items():
    rows.append({
        "Model / input": label,
        **model_info[label],
        **summarize_metrics(predictions, test_targets),
    })

columns = [
    "Model / input",
    "Training set",
    "Backbone",
    "Epochs",
    "Batch size",
    "Weight decay",
    "Learning rate",
    "Checkpoint",
    "Images",
    "SSIM",
    "NRMSE",
    "PSNR (dB)",
]

No working directory provided. Using current working directory: /storage/research/dsl_vibe_rs/trainings/2026_deep_learning_with_VIBE/CAREamics/scripts.


Predicting with U2OS_splits_n100_seed42_CARE-finetuned_n_epochs10_b_size64_w_decay01_l_rate1e-05 / last


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 1g.24gb') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [MIG-871230e5-4f84-5f7b-a288-f2c24ae0169b]
SLURM auto-requeueing enabled. Setting signal handlers.
/opt/.pixi/envs/napari-careamics-default/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'predict_dataloader' d

Predicting DataLoader 0: |                       | 10/? [00:00<00:00, 47.75it/s]


No working directory provided. Using current working directory: /storage/research/dsl_vibe_rs/trainings/2026_deep_learning_with_VIBE/CAREamics/scripts.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [MIG-871230e5-4f84-5f7b-a288-f2c24ae0169b]
SLURM auto-requeueing enabled. Setting signal handlers.


Predicting with U2OS_splits_n100_seed42_CARE-finetuned_n_epochs30_b_size128_w_decay01_l_rate1e-05 / last
Predicting DataLoader 0: |                      | 10/? [00:00<00:00, 307.33it/s]


No working directory provided. Using current working directory: /storage/research/dsl_vibe_rs/trainings/2026_deep_learning_with_VIBE/CAREamics/scripts.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [MIG-871230e5-4f84-5f7b-a288-f2c24ae0169b]
SLURM auto-requeueing enabled. Setting signal handlers.


Predicting with U2OS_splits_n100_seed42_CARE-finetuned_n_epochs30_b_size64_w_decay01_l_rate1e-05 / last
Predicting DataLoader 0: |                      | 10/? [00:00<00:00, 224.71it/s]


## 6. Summarize metrics in a comparison table
Higher SSIM/PSNR and lower NRMSE indicate better agreement.
Metrics are calculated per image, then shown as mean +/- standard deviation.

Because intensities are matched per image, use these metrics to compare restoration quality, not absolute intensity calibration.

In [7]:
display(pd.DataFrame(rows, columns=columns))

,Model / input,Training set,Backbone,Epochs,Batch size,Weight decay,Learning rate,Checkpoint,Images,SSIM,NRMSE,PSNR (dB)
0,Noisy input,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10,-0.017 +/- 0.014,0.177 +/- 0.040,15.242 +/- 1.992
1,U2OS_splits_n100_seed42_CARE-finetuned_n_epoch...,U2OS_splits_n100_seed42,CARE,10.0,64.0,0.1,0.00001,last,10,-0.078 +/- 0.054,0.130 +/- 0.032,18.016 +/- 2.256
2,U2OS_splits_n100_seed42_CARE-finetuned_n_epoch...,U2OS_splits_n100_seed42,CARE,30.0,128.0,0.1,0.00001,last,10,-0.056 +/- 0.049,0.126 +/- 0.031,18.305 +/- 2.261
3,U2OS_splits_n100_seed42_CARE-finetuned_n_epoch...,U2OS_splits_n100_seed42,CARE,30.0,64.0,0.1,0.00001,last,10,-0.052 +/- 0.049,0.126 +/- 0.031,18.274 +/- 2.255


## References:


- CAREamics: https://careamics.github.io/latest/
- Weigert et al., "Content-aware image restoration: pushing the limits of fluorescence microscopy", Nature Methods (2018).
- von Chamier et al., "Democratising deep learning for microscopy with ZeroCostDL4Mic", Nature Communications (2021).
- scikit-image metric definitions: https://scikit-image.org/docs/stable/api/skimage.metrics.html